# One-Shot Prompt PDF RAG with Groq

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhelamay/CPUT_ICAILY_AI2026_MaganLLM_WORKSHOP/blob/main/CPUT_ICAILYAI_092026_PROMPTSENG_RAG/rag1_simple_pdf_one_shot_colab.ipynb)

This notebook matches `rag1_simple_pdf_one_shot.py`.
It teaches one-shot prompting by giving the model a mini example before the real question.

In [ ]:
!pip -q install openai python-dotenv numpy pypdf tiktoken fastembed

In [ ]:
import os
from getpass import getpass

import numpy as np
import tiktoken
from fastembed import TextEmbedding
from openai import OpenAI
from pypdf import PdfReader

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_file_path = next(iter(uploaded.keys())) if uploaded else "sample_ocr.pdf"
except ImportError:
    pdf_file_path = input("PDF path (default sample_ocr.pdf): ").strip() or "sample_ocr.pdf"

groq_api_key = os.getenv("GROQ_API_KEY") or getpass("Enter GROQ_API_KEY: ").strip()
chat_model = os.getenv("GROQ_CHAT_MODEL") or "qwen/qwen3.8-27b"
user_question = input("Question: ").strip() or "Extract the most important business details from this invoice."

client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key)
embedder = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    pages = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages.append(text.strip())
    return "\n".join(pages)

def chunk_text(text: str, chunk_size: int = 500, chunk_overlap: int = 50) -> list[str]:
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)
    step = chunk_size - chunk_overlap
    return [tokenizer.decode(tokens[start:start + chunk_size]) for start in range(0, len(tokens), step)]

def get_embedding(text: str) -> list[float]:
    return list(next(embedder.embed([text])))

def cosine_similarity(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

def retrieve_best_chunk(chunks: list[str], question: str) -> str:
    chunk_embeddings = [get_embedding(chunk) for chunk in chunks]
    question_embedding = get_embedding(question)
    similarities = [cosine_similarity(question_embedding, chunk_embedding) for chunk_embedding in chunk_embeddings]
    return chunks[int(np.argmax(similarities))]

def ask_groq(retrieved_context: str, question: str) -> str:
    example_context = "Proforma Invoice. Seller: ABC Medical Supplies. Buyer: City Hospital. Items: 10 beds at $500 each, total $5,000. Payment: 100% at sight."
    example_question = "Extract the most important business details from this invoice."
    example_answer = "Summary:\n- Document type: Proforma Invoice\n- Seller: ABC Medical Supplies\n- Buyer: City Hospital\n- Main item: 10 beds at $500 each\n- Total amount: $5,000\n- Payment term: 100% at sight"
    completion = client.chat.completions.create(
        model=chat_model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Use only the provided context and follow the answer style shown in the example."},
            {"role": "user", "content": f"Context:\n{example_context}\n\nQuestion:\n{example_question}"},
            {"role": "assistant", "content": example_answer},
            {"role": "user", "content": f"Context:\n{retrieved_context}\n\nQuestion:\n{question}"},
        ],
    )
    return completion.choices[0].message.content or ""

In [ ]:
raw_pdf_text = extract_text_from_pdf(pdf_file_path)
chunks = chunk_text(raw_pdf_text)
retrieved_context = retrieve_best_chunk(chunks, user_question)
answer = ask_groq(retrieved_context, user_question)
print(answer)